## Dirección de MLflow actualizada el 7 de noviembre de 2026

Jairo comunicó como dirección definitiva `http://54.163.35.181:8050/`. La celda de configuración usa esa dirección. Las salidas guardadas se conservan como evidencia histórica de la ejecución anterior al cambio de IP y sus enlaces reflejan el servidor usado entonces. Este notebook de Iris no se volvió a ejecutar durante la actualización.

El experimento técnico continúa siendo `Test-MLflow`. Para consultar el experimento de FINRISK sin crear corridas, seguir `docs/entrega2/mlflow.md`.


In [1]:
%pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. CONFIGURACIÓN DE MLFLOW
# ============================================================

MLFLOW_URI = "http://54.163.35.181:8050"
EXPERIMENT_NAME = "Test-MLflow"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Servidor MLflow: {mlflow.get_tracking_uri()}")
print(f"Experimento: {EXPERIMENT_NAME}")


# ============================================================
# 2. PREPARACIÓN DE DATOS
# ============================================================

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 3. FUNCIÓN DE ENTRENAMIENTO Y REGISTRO EN MLFLOW
# ============================================================

def entrenar_modelo(
    n_estimators=100,
    max_depth=5,
    max_features="sqrt",
    min_samples_leaf=1
):

    # Crear modelo con los hiperparámetros recibidos
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    # Entrenar
    model.fit(X_train, y_train)

    # Predecir
    y_pred = model.predict(X_test)

    # Calcular métricas
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test, y_pred, average="weighted"
        ),
        "recall": recall_score(
            y_test, y_pred, average="weighted"
        ),
        "f1_score": f1_score(
            y_test, y_pred, average="weighted"
        )
    }

    # Hiperparámetros utilizados
    params = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "max_features": max_features,
        "min_samples_leaf": min_samples_leaf,
        "random_state": 42,
        "test_size": 0.2
    }

    # Nombre descriptivo de la ejecución
    run_name = (
        f"RF_n{n_estimators}"
        f"_d{max_depth}"
        f"_f{max_features}"
        f"_leaf{min_samples_leaf}"
    )

    # Registrar ejecución en MLflow
    with mlflow.start_run(run_name=run_name) as run:

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        mlflow.sklearn.log_model(
            sk_model=model,
            name="modelo_iris",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        print(f"\nExperimento registrado: {run_name}")
        print(f"Run ID: {run.info.run_id}")

    # Mostrar resultados
    print("\nMétricas:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

    return metrics

Servidor MLflow: http://52.91.138.185:8050
Experimento: Test-MLflow


In [6]:
resultados = entrenar_modelo(
    n_estimators=200,
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=1
)


Experimento registrado: RF_n200_d10_fsqrt_leaf1
Run ID: e4ef72c1eadc4409bf9c279f79972701
🏃 View run RF_n200_d10_fsqrt_leaf1 at: http://52.91.138.185:8050/#/experiments/1/runs/e4ef72c1eadc4409bf9c279f79972701
🧪 View experiment at: http://52.91.138.185:8050/#/experiments/1

Métricas:
accuracy: 0.9000
precision: 0.9024
recall: 0.9000
f1_score: 0.8997


In [7]:
resultados = entrenar_modelo(
    n_estimators=500,
    max_depth=3,
    max_features="log2",
    min_samples_leaf=2
)


Experimento registrado: RF_n500_d3_flog2_leaf2
Run ID: 4ade0fc3b6e9435da3cb45333ac1f351
🏃 View run RF_n500_d3_flog2_leaf2 at: http://52.91.138.185:8050/#/experiments/1/runs/4ade0fc3b6e9435da3cb45333ac1f351
🧪 View experiment at: http://52.91.138.185:8050/#/experiments/1

Métricas:
accuracy: 0.9667
precision: 0.9697
recall: 0.9667
f1_score: 0.9666
